# TfL New Underground Station Analysis

**LSESU Datathon 2026** â€” Advising TfL on the optimal location for a new Underground station using data-driven spatial analysis.

**Authors:** [Team Name]  
**Date:** March 2026

## Section 0: Setup

In [2]:
import sys
from pathlib import Path

# Add project root to path so we can import src modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')  # must come before any other matplotlib import
import matplotlib.pyplot as plt
import seaborn as sns
import gc

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

In [3]:
from src.config import *
from src.data_download import download_all
from src.data_loader import (
    load_lsoa_boundaries, load_station_locations, load_ptal_lsoa,
    load_imd_scores, load_census_pop_density, load_census_economic_activity,
    load_crowding_data,
)
from src.spatial import build_master_geodataframe
from src.visualisation import (
    plot_distance_to_station, plot_ptal_scores, plot_population_density,
)

print("Imports OK")

Imports OK


## Section 1: Spatial Framework & Data Ingestion (Stage 1)

Build the master LSOA GeoDataFrame that all subsequent stages depend on. This includes:
- Downloading available datasets
- Loading and standardising all data sources
- Computing distance to nearest station and PTAL scores
- Producing choropleth visualisations (V1-V3)

### 1.1 Download Datasets

In [4]:
# Download datasets with stable URLs (idempotent â€” skips if already present)
downloaded = download_all()

  [skip] Station locations already exist: tfl_stations.geojson
  [skip] IMD 2019 scores (File 7, CSV) already exists: imd_2019.csv
  [skip] Station crowding entry/exit (2023) already exists: AC2023_AnnualisedEntryExit.xlsx
  [skip] LSOA boundaries already exist
Downloaded 4/4 datasets.

Manual downloads still needed (see README.md):
  - PTAL grid 2023 (TfL GIS Open Data Hub)
    https://gis-tfl.opendata.arcgis.com/datasets/0646faf45243463aa04ca685e598f471
    Or use LSOA-aggregated version (saves spatial join):
    https://gis-tfl.opendata.arcgis.com/datasets/3eb38b75667a49df9ef1240e9a197615
  - Census 2021 population & economic activity (NOMIS bulk)
    https://www.nomisweb.co.uk/sources/census_2021_bulk


### 1.2 Load Data

In [5]:
# Load station locations (always available after download)
stations = load_station_locations()
stations.head()

Skipping field modes: unsupported OGR type: 5


Loaded 2640 station locations


,id,name,zone,geometry
0,4900ZZDLABR0,Abbey Road DLR Station,,POINT (539086.073 183367.317)
1,4900ZZDLABR1,Abbey Road DLR Station,,POINT (539075.199 183434.902)
2,9400ZZDLABR,Abbey Road DLR Station,,POINT (539080.122 183348.903)
3,9400ZZDLABR1,Abbey Road DLR Station,,POINT (539087.13 183348.984)
4,9400ZZDLABR2,Abbey Road DLR Station,,POINT (539073.111 183348.934)


In [6]:
# Load LSOA boundaries (requires manual download â€” see README)
lsoa = load_lsoa_boundaries()
print(f"LSOA columns: {list(lsoa.columns)}")
lsoa.head()

Loaded 4835 LSOA boundaries (2011 fallback) from LSOA_2011_London_gen_MHW.shp
LSOA columns: ['lsoa_code', 'lsoa_name', 'MSOA11CD', 'MSOA11NM', 'LAD11CD', 'LAD11NM', 'RGN11CD', 'RGN11NM', 'USUALRES', 'HHOLDRES', 'COMESTRES', 'POPDEN', 'HHOLDS', 'AVHHOLDSZ', 'geometry']


,lsoa_code,lsoa_name,MSOA11CD,MSOA11NM,LAD11CD,LAD11NM,RGN11CD,RGN11NM,USUALRES,HHOLDRES,COMESTRES,POPDEN,HHOLDS,AVHHOLDSZ,geometry
0,E01000001,City of London 001A,E02000001,City of London 001,E09000001,City of London,E12000007,London,1465,1465,0,112.9,876,1.7,"POLYGON ((532105.092 182011.23, 532162.491 181..."
1,E01000002,City of London 001B,E02000001,City of London 001,E09000001,City of London,E12000007,London,1436,1436,0,62.9,830,1.7,"POLYGON ((532746.813 181786.891, 532671.688 18..."
2,E01000003,City of London 001C,E02000001,City of London 001,E09000001,City of London,E12000007,London,1346,1250,96,227.7,817,1.5,"POLYGON ((532135.145 182198.119, 532158.25 182..."
3,E01000005,City of London 001E,E02000001,City of London 001,E09000001,City of London,E12000007,London,985,985,0,52.0,467,2.1,"POLYGON ((533807.946 180767.77, 533649.063 180..."
4,E01000006,Barking and Dagenham 016A,E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E12000007,London,1703,1699,4,116.2,543,3.1,"POLYGON ((545122.049 184314.931, 545271.917 18..."


In [7]:
# Load optional datasets â€” wrapped in try/except so notebook still runs
# if manual downloads haven't been completed yet

ptal = None
try:
    ptal = load_ptal_lsoa()
except FileNotFoundError as e:
    print(f"PTAL data not available: {e}")

imd = None
try:
    imd = load_imd_scores()
except FileNotFoundError as e:
    print(f"IMD scores not available: {e}")

census_popden = None
try:
    census_popden = load_census_pop_density()
except FileNotFoundError as e:
    print(f"Census population density not available: {e}")

census_econ = None
try:
    census_econ = load_census_economic_activity()
except FileNotFoundError as e:
    print(f"Census economic activity not available: {e}")

crowding = None
try:
    crowding = load_crowding_data()
except FileNotFoundError as e:
    print(f"Crowding data not available: {e}")


Loaded PTAL stats for 4994 LSOAs
Loaded IMD scores for 32844 LSOAs
Census population density not available: [Errno 2] No such file or directory: 'C:\\Users\\chang\\Data Science Projects\\big-d\\data\\raw\\census2021-ts006-lsoa-populationdensity.csv'
Census economic activity not available: [Errno 2] No such file or directory: 'C:\\Users\\chang\\Data Science Projects\\big-d\\data\\raw\\census2021-ts066-lsoa-economicactivity.csv'
Loaded crowding data from AC2023_AnnualisedEntryExit.xlsx (430 stations)


### 1.3 Build Master GeoDataFrame

In [8]:
# Build master LSOA GeoDataFrame with all joined data
master = build_master_geodataframe(
    lsoa_gdf=lsoa,
    stations_gdf=stations,
    ptal_df=ptal,
    imd_df=imd,
    census_popden_df=census_popden,
    census_econ_df=census_econ,
    crowding_df=crowding,
    save=True,
)
print(f"\nMaster GDF shape: {master.shape}")
print(f"Columns: {list(master.columns)}")
master.head()

Computing distance to nearest station ...
Joining PTAL scores ...
Joining IMD scores ...
Computing crowding pressure ...
Crowding joined: 408/470 stations matched (86.8%)
Saved master GeoDataFrame to C:\Users\chang\Data Science Projects\big-d\data\processed\master_lsoa.gpkg (4835 rows)

Master GDF shape: (4835, 80)
Columns: ['lsoa_code', 'lsoa_name', 'MSOA11CD', 'MSOA11NM', 'LAD11CD', 'LAD11NM', 'RGN11CD', 'RGN11NM', 'USUALRES', 'HHOLDRES', 'COMESTRES', 'POPDEN', 'HHOLDS', 'AVHHOLDSZ', 'geometry', 'dist_to_station_m', 'mean_ptal_ai', 'median_ptal_ai', 'min_ptal_ai', 'max_ptal_ai', 'mean_ptal_level', 'LSOA name (2011)', 'Local Authority District code (2019)', 'Local Authority District name (2019)', 'imd_score', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Score (rate)', 'Income Rank (where 1 is most deprived)', 'Income Decile (where 1 is most deprived 10% of LSOAs)', '

,lsoa_code,lsoa_name,MSOA11CD,MSOA11NM,LAD11CD,LAD11NM,RGN11CD,RGN11NM,USUALRES,HHOLDRES,COMESTRES,POPDEN,HHOLDS,AVHHOLDSZ,geometry,dist_to_station_m,mean_ptal_ai,median_ptal_ai,min_ptal_ai,max_ptal_ai,...,Geographical Barriers Sub-domain Score,Geographical Barriers Sub-domain Rank (where 1 is most deprived),Geographical Barriers Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Wider Barriers Sub-domain Score,Wider Barriers Sub-domain Rank (where 1 is most deprived),Wider Barriers Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners),nearest_station_ann_total,crowding_pressure,area_km2
0,E01000001,City of London 001A,E02000001,City of London 001,E09000001,City of London,E12000007,London,1465,1465,0,112.9,876,1.7,"POLYGON ((532105.092 182011.23, 532162.491 181...",261.740388,74.818060,76.150404,43.344686,97.664050,...,-0.430,22985,7,3.587,3216,1,0.006,16364,5,1.503,1615,1,1296,175,656,465,715,5185508.0,2.984772e+07,0.133321
1,E01000002,City of London 001B,E02000001,City of London 001,E09000001,City of London,E12000007,London,1436,1436,0,62.9,830,1.7,"POLYGON ((532746.813 181786.891, 532671.688 18...",211.983855,85.324898,88.899953,44.802674,102.127724,...,-1.060,30223,10,3.231,3894,2,-0.410,22676,7,1.196,2969,1,1156,182,580,394,620,23330204.0,4.361514e+07,0.226191
2,E01000003,City of London 001C,E02000001,City of London 001,E09000001,City of London,E12000007,London,1346,1250,96,227.7,817,1.5,"POLYGON ((532135.145 182198.119, 532158.25 182...",198.466652,53.411383,54.114541,36.995870,66.989479,...,-0.691,26719,9,5.173,818,1,-0.054,17318,6,2.207,162,1,1350,146,759,445,804,5185508.0,2.675734e+07,0.057303
3,E01000005,City of London 001E,E02000001,City of London 001,E09000001,City of London,E12000007,London,985,985,0,52.0,467,2.1,"POLYGON ((533807.946 180767.77, 533649.063 180...",60.667490,82.428912,86.649985,45.504887,104.154096,...,-1.167,30865,10,5.361,672,1,-0.604,25218,8,1.769,849,1,1121,229,692,200,683,6897314.0,3.496323e+07,0.190739
4,E01000006,Barking and Dagenham 016A,E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E12000007,London,1703,1699,4,116.2,543,3.1,"POLYGON ((545122.049 184314.931, 545271.917 18...",490.908349,24.153971,27.931273,2.836977,33.400903,...,-0.400,22527,7,5.590,520,1,0.110,14745,5,0.969,4368,2,2040,522,1297,221,1285,15137045.0,5.655665e+06,0.144196


In [9]:
# Summary statistics for key columns
summary_cols = [c for c in [
    "dist_to_station_m", "mean_ptal_ai", "imd_score",
    "pop_density_2021", "employment_rate",
    "nearest_station_ann_total", "crowding_pressure", "area_km2",
] if c in master.columns]
master[summary_cols].describe()

,dist_to_station_m,mean_ptal_ai,imd_score,nearest_station_ann_total,crowding_pressure,area_km2
count,4835.000000,4659.000000,4835.000000,4.835000e+03,4.835000e+03,4835.000000
mean,1583.571145,12.689941,21.498424,5.885095e+06,2.544292e+06,0.325441
std,2009.673718,11.064015,10.904741,6.316243e+06,5.971463e+06,0.629009
min,8.974539,0.213923,2.326000,2.676790e+05,0.000000e+00,0.016902
25%,445.266073,5.660945,12.446500,2.223811e+06,0.000000e+00,0.133548
50%,825.543336,9.386386,20.375000,4.040066e+06,5.024462e+05,0.203355
75%,1726.469778,16.141646,29.597500,7.948900e+06,2.594356e+06,0.319056
max,14532.009271,98.327599,64.677000,7.212426e+07,7.944372e+07,15.808727


### 1.4 Stage 1 Visualisations

In [10]:
# V1: Distance to nearest station
fig1 = plot_distance_to_station(master)
plt.show()

Saved C:\Users\chang\Data Science Projects\big-d\figures\v1_distance_to_station.png


In [11]:
# V2: PTAL scores (only if PTAL data was loaded)
if "mean_ptal_ai" in master.columns and master["mean_ptal_ai"].notna().any():
    fig2 = plot_ptal_scores(master)
    plt.show()
else:
    print("Skipping V2: PTAL data not available")

Saved C:\Users\chang\Data Science Projects\big-d\figures\v2_ptal_scores.png


In [12]:
# V3: Population density (only if census data was loaded)
if "pop_density_km2" in master.columns and master["pop_density_km2"].notna().any():
    fig3 = plot_population_density(master)
    plt.show()
else:
    print("Skipping V3: Census population data not available")

Skipping V3: Census population data not available


In [13]:
# V4: Station crowding pressure
if "crowding_pressure" in master.columns and master["crowding_pressure"].notna().any():
    fig4, ax = plt.subplots(1, 1, figsize=(12, 10))
    master.plot(
        column="crowding_pressure",
        cmap="Reds",
        legend=True,
        legend_kwds={"label": "Crowding Pressure (weighted annual entries)"},
        ax=ax,
        missing_kwds={"color": "lightgrey"},
    )
    ax.set_title("Station Crowding Pressure by LSOA (2023)", fontsize=14)
    ax.set_axis_off()
    plt.tight_layout()
    fig4.savefig(FIGURES_DIR / "v4_crowding_pressure.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping V4: Crowding data not available")

---

## Section 2: Demand & Deprivation Scoring (Stage 2)

- Combine population density, IMD deprivation, and economic activity into a composite demand score
- Weight sub-indicators using PCA or domain knowledge
- Identify LSOAs with high unmet demand (high population + high deprivation + low PTAL)

Based on the LSOA explorer HTML, we can output a ranking of the regions in London that need a Tube station. We do this and save the file under as ```data/station_need_scores.csv```

## Section 3: Network Graph Analysis (Stage 3)

- Build Tube/rail network as a weighted graph (NetworkX)
- Compute betweenness centrality, closeness centrality, and degree
- Identify network gaps: areas far from high-centrality nodes
- Evaluate candidate locations for network connectivity improvement

In [14]:
import numpy as np
import networkx as nx
from matplotlib.colors import LinearSegmentedColormap
from collections import defaultdict
import ast, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'figure.dpi': 150
})

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance (km). Supports numpy broadcasting."""
    R = 6371
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

print("Environment ready.")

Environment ready.


### 3.1. Load Data

### 3.1.a. TfL Network Data
We load the real station and edge data from `tfl-node-sim-main/`. This gives us:
- **358 stations** with official TfL station IDs, coordinates, and lines served
- **492 edges** with pre-computed distances in km, including 25 explicit interchange links
- All 13 lines: Bakerloo, Central, Circle, District, DLR, Elizabeth, Hammersmith & City, Jubilee, Metropolitan, Northern, Piccadilly, Victoria, Waterloo & City

In [15]:
# === Load TfL station data ===
# Update these paths to match your local file locations
STATIONS_PATH = '../data/tube_stations_fixed.csv'
EDGES_PATH = '../data/tube_edges_fixed.csv'
LSOA_GEO_PATH = '../data/london_lsoa_2021.geojson'
MASTER_LSOA_PATH = '../data/master_lsoa.gpkg'
NEED_SCORES_PATH = '../data/station_need_scores.csv'

stations = pd.read_csv(STATIONS_PATH)
edges = pd.read_csv(EDGES_PATH)

# Parse the lines_served column from string representation of list
stations['lines_list'] = stations['lines_served'].apply(ast.literal_eval)

print(f"Stations loaded: {len(stations)}")
print(f"Edges loaded: {len(edges)}")
print(f"  - Line edges: {len(edges[edges['line'] != 'interchange'])}")
print(f"  - Interchange edges: {len(edges[edges['line'] == 'interchange'])}")

# Show all lines and station counts
all_lines = set()
for lines in stations['lines_list']:
    all_lines.update(lines)
print(f"\nLines ({len(all_lines)}): {sorted(all_lines)}")

print(f"\nMulti-line interchanges: {(stations['num_lines'] > 1).sum()} stations")
print(f"Max lines at one station: {stations['num_lines'].max()}")

Stations loaded: 358
Edges loaded: 492
  - Line edges: 467
  - Interchange edges: 25

Lines (13): ['bakerloo', 'central', 'circle', 'district', 'dlr', 'elizabeth', 'hammersmith-city', 'jubilee', 'metropolitan', 'northern', 'piccadilly', 'victoria', 'waterloo-city']

Multi-line interchanges: 104 stations
Max lines at one station: 6


In [16]:
# Quick preview of the data
print("=== Sample Stations ===")
print(stations[['station_id','station_name','lat','lon','lines_served','num_lines']].head(10).to_string())
print(f"\n=== Sample Edges ===")
print(edges[['station_from_name','station_to_name','line','edge_distance_km']].head(10).to_string())
print(f"\n=== Interchange Edges ===")
ic = edges[edges['line'] == 'interchange']
print(ic[['station_from_name','station_to_name','edge_distance_km']].to_string())

=== Sample Stations ===
    station_id                      station_name        lat       lon                      lines_served  num_lines
0  940GZZDLABR            Abbey Road DLR Station  51.531926  0.003737                           ['dlr']          1
1   910GABWDXR                        Abbey Wood  51.491284  0.121087                     ['elizabeth']          1
2  910GACTONML      Acton Main Line Rail Station  51.517180 -0.266756                     ['elizabeth']          1
3  940GZZLUACT    Acton Town Underground Station  51.503057 -0.280462        ['district', 'piccadilly']          2
4  940GZZLUADE  Aldgate East Underground Station  51.515037 -0.072384  ['district', 'hammersmith-city']          2
5  940GZZLUALD       Aldgate Underground Station  51.514246 -0.075689        ['circle', 'metropolitan']          2
6  940GZZDLALL            All Saints DLR Station  51.511000 -0.013135                           ['dlr']          1
7  940GZZLUALP      Alperton Underground Station  51.540

### 3.1.b. LSOA & Need Score Data
We load LSOA-level socioeconomic data (population, IMD, PTAL) and composite need scores.

In [17]:
# Load LSOA master data (for catchment population, IMD)
gdf = gpd.read_file(MASTER_LSOA_PATH)
print(f"Master LSOA data: {len(gdf)} LSOAs, {len(gdf.columns)} columns")

# Load need scores
needs = pd.read_csv(NEED_SCORES_PATH)
print(f"Need scores: {len(needs)} LSOAs")
print(f"Top-10 flagged: {needs['is_top10_location'].sum()}")

# Show flagged candidates
top_flagged = needs[needs['is_top10_location']].sort_values('rank').copy()
print(f"\n{'Rank':>4}  {'LSOA Name':<30} {'Borough':<20} {'Score':>6}")
print("-" * 70)
for _, r in top_flagged.iterrows():
    la = r.get('la_name', '') or ''
    print(f"{r['rank']:>4}  {r['lsoa_name']:<30} {la:<20} {r['composite_score']:>6.4f}")

Master LSOA data: 4994 LSOAs, 115 columns
Need scores: 4994 LSOAs
Top-10 flagged: 10

Rank  LSOA Name                      Borough               Score
----------------------------------------------------------------------
   1  Brent 035C                     nan                  0.5312
   2  Tower Hamlets 034A             nan                  0.4912
   3  Bromley 032B                   Bromley              0.4798
   8  Waltham Forest 028C            Waltham Forest       0.4552
  14  Lewisham 001A                  Lewisham             0.4358
  15  Hackney 007E                   Hackney              0.4349
  20  Lambeth 004G                   Lambeth              0.4187
  21  Camden 015D                    Camden               0.4184
  23  Islington 001C                 Islington            0.4116
  24  Lambeth 011F                   Lambeth              0.4108


### 3.2. Build the Network Graph from TfL Data

We construct a NetworkX graph directly from the edge list. Each station becomes a node (keyed by `station_id`), and each row in `tube_edges_fixed.csv` becomes a weighted edge. The `edge_distance_km` column provides pre-computed distances — much more accurate than Haversine approximations between centroids.

**Importantly**, this data includes **interchange edges** (line = `'interchange'`) that explicitly model the walking time between different platforms at interchange stations (e.g., Bank ↔ Monument, Paddington ↔ Paddington H&C).

In [18]:
# Build a station lookup dictionary: station_id -> {name, lat, lon, lines}
sdict = {}
for _, row in stations.iterrows():
    sdict[row['station_id']] = {
        'name': row['station_name'],
        'lat': row['lat'],
        'lon': row['lon'],
        'lines': row['lines_list'],
        'num_lines': row['num_lines']
    }

# Build the graph   
G = nx.Graph()

# Add nodes
for sid, info in sdict.items():
    G.add_node(sid, name=info['name'], lat=info['lat'], lon=info['lon'],
               lines=info['lines'], num_lines=info['num_lines'])

# Add edges from the edge list
for _, row in edges.iterrows():
    s1, s2 = row['station_from_id'], row['station_to_id']
    line = row['line']
    dist = row['edge_distance_km']
    
    if G.has_edge(s1, s2):
        # Edge already exists (multi-line) — keep shorter distance, add line
        G[s1][s2]['lines'].add(line)
        G[s1][s2]['weight'] = min(G[s1][s2]['weight'], dist)
    else:
        G.add_edge(s1, s2, weight=dist, lines={line})

# Check connectivity
print(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected: {nx.is_connected(G)}")

if not nx.is_connected(G):
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    print(f"Components: {[len(c) for c in components[:5]]}")
    G = G.subgraph(components[0]).copy()
    print(f"Using largest component: {G.number_of_nodes()} nodes")
else:
    print("✓ Fully connected graph — all stations reachable")

Network: 358 nodes, 427 edges
Connected: True
✓ Fully connected graph — all stations reachable


### 3.3. Baseline Network Metrics

We compute reference metrics before any insertions:

- **Global efficiency**: $E = \frac{1}{N(N-1)} \sum_{i \neq j} \frac{1}{d_{ij}}$
- **Average shortest path length**: mean distance (km) across all station pairs
- **Betweenness centrality**: $C_B(v) = \sum_{s \neq t \neq v} \frac{\sigma_{st}(v)}{\sigma_{st}}$ — fraction of all shortest paths passing through each station

In [19]:

bc_base = nx.betweenness_centrality(G, weight='weight')
ge_base = nx.global_efficiency(G)
apl_base = nx.average_shortest_path_length(G, weight='weight')

print(f"Global efficiency (baseline):   {ge_base:.6f}")
print(f"Avg shortest path (baseline):   {apl_base:.2f} km")
print(f"\nTop 15 bottleneck stations (betweenness centrality):")
print(f"{'Station':<50} {'BC':>8}")
print("-" * 60)
for sid, val in sorted(bc_base.items(), key=lambda x: -x[1])[:15]:
    print(f"{G.nodes[sid]['name']:<50} {val:>8.4f}")

Global efficiency (baseline):   0.103615
Avg shortest path (baseline):   19.99 km

Top 15 bottleneck stations (betweenness centrality):
Station                                                  BC
------------------------------------------------------------
Liverpool Street                                     0.2470
Bank Underground Station                             0.2406
King's Cross St. Pancras Underground Station         0.2098
Baker Street Underground Station                     0.1904
Farringdon                                           0.1736
Earl's Court Underground Station                     0.1705
Stratford (London) Rail Station                      0.1647
Bond Street Underground Station                      0.1491
South Kensington Underground Station                 0.1472
Gloucester Road Underground Station                  0.1456
Bond Street                                          0.1445
Oxford Circus Underground Station                    0.1436
Paddington             

### 3.4. Identify Feasible Lines for Each Candidate

For each flagged location, we find all stations within 4 km, determine which lines serve those stations, and pick the nearest station per line as the insertion point.

In [20]:

def find_connections(lat, lon, G, sdict, max_d=4.0):
    """Find nearest station on each line within max_d km of a candidate location."""
    nearby = []
    for sid, info in sdict.items():
        if sid not in G.nodes:
            continue
        d = haversine_km(lat, lon, info['lat'], info['lon'])
        if d <= max_d:
            for line in info['lines']:
                nearby.append({'station_id': sid, 'station': info['name'],
                              'line': line, 'dist': d})
    if not nearby:
        return []
    df = pd.DataFrame(nearby)
    # For each line, keep only the nearest station
    return df.loc[df.groupby('line')['dist'].idxmin()].sort_values('dist').head(6).to_dict('records')

# Find connections for each candidate
cands = []
for _, row in top_flagged.iterrows():
    conns = find_connections(row['latitude'], row['longitude'], G, sdict)
    cands.append({
        'lsoa_code': row['lsoa_code'], 'lsoa_name': row['lsoa_name'],
        'la_name': row.get('la_name', ''), 'lat': row['latitude'], 'lon': row['longitude'],
        'need_score': row['composite_score'], 'rank': row['rank'], 'conns': conns
    })
    print(f"\n{row['lsoa_name']} ({row['latitude']:.4f}, {row['longitude']:.4f}):")
    if not conns:
        print("  ⚠ No stations within 4 km — cannot connect!")
    for c in conns[:5]:
        print(f"  → {c['line']:<20} via {c['station']:<45} ({c['dist']:.2f} km)")


Brent 035C (51.5585, -0.2849):
  → jubilee              via Wembley Park Underground Station              (0.65 km)
  → metropolitan         via Wembley Park Underground Station              (0.65 km)
  → bakerloo             via Wembley Central Underground Station           (1.08 km)
  → piccadilly           via Alperton Underground Station                  (2.23 km)
  → central              via Hanger Lane Underground Station               (3.20 km)

Tower Hamlets 034A (51.5126, 0.0057):
  → dlr                  via Canning Town Underground Station              (0.21 km)
  → jubilee              via Canning Town Underground Station              (0.21 km)
  → elizabeth            via Custom House Station                          (1.48 km)
  → district             via West Ham DLR Station                          (1.70 km)
  → hammersmith-city     via West Ham DLR Station                          (1.70 km)

Bromley 032B (51.3759, 0.1193):
  ⚠ No stations within 4 km — cannot connect!


## Section 4: Candidate Site Identification (Stage 4)

- Apply spatial filters: minimum distance from existing stations, within Greater London
- Score candidates using weighted multi-criteria model (demand, connectivity, deprivation)
- Cluster high-scoring LSOAs into candidate zones
- Shortlist top 5-10 candidate sites for detailed evaluation

### Simulate Station Insertion & Compute Impact Metrics

For each candidate we:
1. **Insert** a new node into the graph
2. **Connect** it to the nearest station(s) on its feasible lines, plus their graph neighbours (so it sits *between* existing stations rather than dangling)
3. **Recompute** all network metrics and measure the delta

### Metrics per candidate

| Metric | Description |
|--------|-------------|
| ΔGlobal Efficiency | % improvement in network-wide efficiency |
| Bottleneck Relief | Reduction in maximum betweenness centrality |
| Improved OD Pairs | Station pairs that get a shorter path |
| Path Savings (km) | Total km saved across improved pairs |
| New Station BC | Centrality of the inserted node |
| Catchment Population | People within 800m (from LSOA data) |
| Catchment IMD | Average deprivation within 800m |

In [21]:
def simulate(G, cand, sdict, gdf, bc_base, ge_base, apl_base):
    """Insert candidate station into graph, compute all impact metrics.
    
    Now tracks the exact insertion topology: for each line, records which
    two existing stations the new station is inserted between, or whether
    it's a terminal extension.
    """
    lat, lon = cand['lat'], cand['lon']
    nn = f"NEW_{cand['lsoa_code']}"
    if not cand['conns']:
        return None
    
    Gn = G.copy()
    Gn.add_node(nn, name=f"New: {cand['lsoa_name']}", lat=lat, lon=lon, lines=[], num_lines=0)
    connected = set()
    lines_s = set()
    
    # Track insertion topology: list of dicts describing where the station sits
    insertions = []
    
    for conn in cand['conns'][:3]:  # Connect via up to 3 lines
        sid = conn['station_id']
        line = conn['line']
        d = conn['dist']
        stn_name = G.nodes[sid]['name'] if sid in G.nodes else sid
        
        if sid not in connected:
            Gn.add_edge(nn, sid, weight=d, lines={line})
            connected.add(sid)
            lines_s.add(line)
        
        # Find the neighbour of this station on the SAME line
        # so the new station sits between them: [neighbour] -- [NEW] -- [sid]
        inserted_between = None
        for neighbor in G.neighbors(sid):
            if neighbor in connected:
                continue
            edge_data = G[sid][neighbor]
            if line in edge_data.get('lines', set()):
                d2 = haversine_km(lat, lon, G.nodes[neighbor]['lat'], G.nodes[neighbor]['lon'])
                if d2 < d * 2.5:  # Only if reasonably close
                    neighbor_name = G.nodes[neighbor]['name']
                    old_edge_dist = edge_data['weight']
                    
                    Gn.add_edge(nn, neighbor, weight=d2, lines={line})
                    connected.add(neighbor)
                    
                    # Record: new station inserted between sid and neighbor on this line
                    insertions.append({
                        'line': line,
                        'type': 'between',
                        'station_a_id': sid,
                        'station_a': stn_name,
                        'dist_to_a_km': round(d, 3),
                        'station_b_id': neighbor,
                        'station_b': neighbor_name,
                        'dist_to_b_km': round(d2, 3),
                        'old_edge_km': round(old_edge_dist, 3),
                        'description': f"{stn_name}  ←{d:.2f}km→  [NEW]  ←{d2:.2f}km→  {neighbor_name}"
                    })
                    inserted_between = neighbor
                    break  # One neighbour per line
        
        # If no same-line neighbour found, this is a terminal extension
        if inserted_between is None:
            insertions.append({
                'line': line,
                'type': 'extension',
                'station_a_id': sid,
                'station_a': stn_name,
                'dist_to_a_km': round(d, 3),
                'station_b_id': None,
                'station_b': None,
                'dist_to_b_km': None,
                'old_edge_km': None,
                'description': f"{stn_name}  ←{d:.2f}km→  [NEW]  (terminal extension)"
            })
    
    if not connected:
        return None
    
    # --- Recompute metrics ---
    bc_n = nx.betweenness_centrality(Gn, weight='weight')
    ge_n = nx.global_efficiency(Gn)
    apl_n = nx.average_shortest_path_length(Gn, weight='weight') if nx.is_connected(Gn) else apl_base

    del Gn
    gc.collect()
    
    max_bc_b = max(bc_base.values())
    max_bc_a = max(v for k, v in bc_n.items() if k != nn)
    
    # Sample OD pair improvements
    imp_pairs = 0; path_sav = 0
    sample = list(G.nodes)[:60]
    for s in sample:
        for t in sample:
            if s >= t: continue
            try:
                od = nx.shortest_path_length(G, s, t, weight='weight')
                nd = nx.shortest_path_length(Gn, s, t, weight='weight')
                if nd < od - 0.01:
                    imp_pairs += 1
                    path_sav += (od - nd)
            except: pass
    
    # Catchment demographics from LSOA data
    cpop = 0; clsoas = 0; cimd = []
    for _, lr in gdf.iterrows():
        d = haversine_km(lat, lon, float(lr['lat']), float(lr['long']))
        if d <= 0.8:
            cpop += lr.get('total_population_mid_2015_excluding_prisoners', 0) or 0
            clsoas += 1
            if pd.notna(lr.get('imd_score')):
                cimd.append(lr['imd_score'])
    
    connected_names = [G.nodes[s]['name'] for s in connected if s in G.nodes]
    
    return {
        'nn': nn, 'lines': list(lines_s), 'connected_ids': list(connected),
        'connected': connected_names, 'n_conn': len(connected),
        'insertions': insertions,  # NEW: full insertion topology
        'bc_new': bc_n.get(nn, 0), 'bottleneck_relief': max_bc_b - max_bc_a,
        'ge_new': ge_n, 'dge': (ge_n - ge_base) / ge_base * 100,
        'apl_new': apl_n, 'dapl': apl_base - apl_n,
        'imp_pairs': imp_pairs, 'path_sav': path_sav,
        'cpop': cpop, 'clsoas': clsoas, 'cimd': np.mean(cimd) if cimd else 0
    }

# Run simulation for all candidates
results = []
for i, c in enumerate(cands):
    print(f"[{i+1}/{len(cands)}] {c['lsoa_name']}...", end=' ')
    m = simulate(G, c, sdict, gdf, bc_base, ge_base, apl_base)
    gc.collect()
    if m is None:
        print("SKIP (no feasible connections)")
        continue
    r = {**{k: v for k, v in c.items() if k != 'conns'}, **m}
    results.append(r)
    print(f"✓  ΔGE={m['dge']:+.4f}%  BR={m['bottleneck_relief']:+.4f}  Pop={m['cpop']}")

rdf = pd.DataFrame(results)
print(f"\nSuccessfully simulated {len(rdf)} candidates")

# === NEW OUTPUT: Show insertion topology for each candidate ===
print("\n" + "=" * 90)
print("INSERTION TOPOLOGY — Where each new station sits in the network")
print("=" * 90)
for _, r in rdf.iterrows():
    print(f"\n  📍 {r['lsoa_name']} ({r['lat']:.4f}, {r['lon']:.4f})")
    for ins in r['insertions']:
        line_label = ins['line'].replace('-', ' ').title()
        if ins['type'] == 'between':
            print(f"     [{line_label} Line]")
            print(f"       BEFORE:  {ins['station_a']}  ←—{ins['old_edge_km']}km—→  {ins['station_b']}")
            print(f"       AFTER:   {ins['station_a']}  ←{ins['dist_to_a_km']}km→  [NEW]  ←{ins['dist_to_b_km']}km→  {ins['station_b']}")
        else:
            print(f"     [{line_label} Line]")
            print(f"       EXTENSION: {ins['station_a']}  ←{ins['dist_to_a_km']}km→  [NEW]  (new terminus)")

[1/10] Brent 035C... ✓  ΔGE=+1.1863%  BR=+0.0004  Pop=nan
[2/10] Tower Hamlets 034A... ✓  ΔGE=+0.2578%  BR=-0.0125  Pop=nan
[3/10] Bromley 032B... SKIP (no feasible connections)
[4/10] Waltham Forest 028C... ✓  ΔGE=+0.7176%  BR=-0.0024  Pop=25915.0
[5/10] Lewisham 001A... ✓  ΔGE=+0.8886%  BR=+0.0067  Pop=nan
[6/10] Hackney 007E... ✓  ΔGE=+0.6769%  BR=+0.0002  Pop=26540.0
[7/10] Lambeth 004G... ✓  ΔGE=+0.4004%  BR=+0.0005  Pop=nan
[8/10] Camden 015D... ✓  ΔGE=+0.3903%  BR=+0.0002  Pop=nan
[9/10] Islington 001C... ✓  ΔGE=+0.6769%  BR=+0.0002  Pop=27970.0
[10/10] Lambeth 011F... ✓  ΔGE=+0.3903%  BR=+0.0005  Pop=31990.0

Successfully simulated 9 candidates

INSERTION TOPOLOGY — Where each new station sits in the network

  📍 Brent 035C (51.5585, -0.2849)
     [Jubilee Line]
       EXTENSION: Wembley Park Underground Station  ←0.652km→  [NEW]  (new terminus)
     [Metropolitan Line]
       EXTENSION: Wembley Park Underground Station  ←0.652km→  [NEW]  (new terminus)
     [Bakerloo Line]
   

## Section 5: Impact Modelling (Stage 5)

- Simulate adding each candidate station to the network
- Recompute PTAL, distance-to-station, and catchment population
- Estimate ridership using gravity model with distance decay
- Quantify improvement in accessibility metrics per candidate

### Multi-Criteria Decision Analysis (MCDA)

We combine network-impact metrics with demand-side need scores using weighted linear combination:

| Criterion | Weight | Source |
|-----------|--------|--------|
| Composite Need Score | 0.20 | Your model |
| Catchment Population | 0.15 | LSOA data |
| Global Efficiency Δ | 0.12 | Network simulation |
| Bottleneck Relief | 0.10 | Network simulation |
| Deprivation (IMD) | 0.10 | LSOA data |
| Improved OD Pairs | 0.08 | Network simulation |
| Path Reduction | 0.08 | Network simulation |
| Network Connections | 0.07 | Feasibility analysis |
| Station Centrality | 0.05 | Network simulation |
| Path Savings (km) | 0.05 | Network simulation |

Each criterion is min-max normalised to [0, 1], then weighted. Total weight sums to 1.

In [22]:
criteria = {
    'dge':               (0.12, 'max', 'Global Eff. Δ'),
    'bottleneck_relief': (0.10, 'max', 'Bottleneck Relief'),
    'imp_pairs':         (0.08, 'max', 'Improved OD Pairs'),
    'dapl':              (0.08, 'max', 'Path Reduction'),
    'bc_new':            (0.05, 'max', 'Station Centrality'),
    'need_score':        (0.20, 'max', 'Need Score'),
    'cpop':              (0.15, 'max', 'Catchment Pop'),
    'cimd':              (0.10, 'max', 'Deprivation'),
    'n_conn':            (0.07, 'max', 'Connections'),
    'path_sav':          (0.05, 'max', 'Path Savings km'),
}
ccols = list(criteria.keys())

# Normalise
for col, (w, d, l) in criteria.items():
    v = rdf[col].values.astype(float)
    mn, mx = v.min(), v.max()
    n = (v - mn) / (mx - mn) if mx - mn > 0 else np.full_like(v, 0.5)
    if d == 'min': n = 1 - n
    rdf[f'{col}_n'] = n

# Weighted composite
tw = sum(w for w, _, _ in criteria.values())
weights = {c: w / tw for c, (w, _, _) in criteria.items()}
rdf['mcda'] = sum(rdf[f'{c}_n'] * w for c, w in weights.items())
rdf = rdf.sort_values('mcda', ascending=False)
rdf['mcda_rank'] = range(1, len(rdf) + 1)

print(f"{'Rank':<5} {'Location':<30} {'MCDA':>7} {'Need':>7} {'ΔGE%':>8} {'BNeck':>7} {'Pop':>7}  Lines")
print("-" * 100)
for _, r in rdf.iterrows():
    ls = ', '.join(r['lines'][:3])
    print(f"{r['mcda_rank']:<5} {r['lsoa_name']:<30} {r['mcda']:>7.4f} {r['need_score']:>7.4f} "
          f"{r['dge']:>+7.4f}% {r['bottleneck_relief']:>+6.4f} {r['cpop']:>7.0f}  {ls}")

Rank  Location                          MCDA    Need     ΔGE%   BNeck     Pop  Lines
----------------------------------------------------------------------------------------------------
1     Brent 035C                      0.6224  0.5312 +1.1863% +0.0004     nan  jubilee, bakerloo
2     Lewisham 001A                   0.5201  0.4358 +0.8886% +0.0067     nan  jubilee, dlr, elizabeth
3     Camden 015D                     0.4442  0.4184 +0.3903% +0.0002     nan  piccadilly, northern
4     Waltham Forest 028C             0.4331  0.4552 +0.7176% -0.0024   25915  dlr, central, elizabeth
5     Tower Hamlets 034A              0.4206  0.4912 +0.2578% -0.0125     nan  dlr, elizabeth
6     Hackney 007E                    0.4128  0.4349 +0.6769% +0.0002   26540  piccadilly, northern
7     Islington 001C                  0.3941  0.4116 +0.6769% +0.0002   27970  piccadilly, northern
8     Lambeth 011F                    0.3810  0.4108 +0.3903% +0.0005   31990  victoria, bakerloo, northern
9     Lam

## Section 6: Sensitivity Analysis (Stage 6)

- Vary key parameters: catchment radius, decay alpha, scoring weights
- Test robustness of top candidates across parameter ranges
- Monte Carlo simulation for uncertainty quantification
- Identify which candidates are robust vs. parameter-sensitive

We test ranking robustness by perturbing all MCDA weights by **±30%** across **10,000 Monte Carlo simulations**. If the winner is stable across most weight combinations, the ranking reflects genuine dominance rather than an artefact of our specific weight choices.

In [23]:
np.random.seed(42)
nsim = 10000
wins = defaultdict(int)
rdist = defaultdict(list)
bw = np.array([weights[c] for c in ccols])

for _ in range(nsim):
    pw = bw * np.random.uniform(0.7, 1.3, len(bw))
    pw /= pw.sum()
    sc = sum(rdf[f'{c}_n'].values * pw[j] for j, c in enumerate(ccols))
    rk = np.argsort(-sc)
    wins[rdf.iloc[rk[0]]['lsoa_name']] += 1
    for k, idx in enumerate(rk):
        rdist[rdf.iloc[idx]['lsoa_name']].append(k + 1)

print(f"Win frequency (±30% perturbation, {nsim:,} simulations):\n")
for n, c in sorted(wins.items(), key=lambda x: -x[1]):
    print(f"  {n:<30} {c/nsim*100:>6.1f}% ({c:>5} wins)")

print(f"\nRank distribution:")
for n in rdf['lsoa_name']:
    rr = rdist[n]
    print(f"  {n:<30} median={np.median(rr):.0f}  mean={np.mean(rr):.1f}  "
          f"P25={np.percentile(rr,25):.0f}  P75={np.percentile(rr,75):.0f}")

Win frequency (±30% perturbation, 10,000 simulations):

  Brent 035C                      100.0% (10000 wins)

Rank distribution:
  Brent 035C                     median=1  mean=1.0  P25=1  P75=1
  Lewisham 001A                  median=2  mean=2.0  P25=2  P75=2
  Camden 015D                    median=3  mean=3.5  P25=3  P75=4
  Waltham Forest 028C            median=4  mean=4.0  P25=4  P75=4
  Tower Hamlets 034A             median=5  mean=5.2  P25=4  P75=6
  Hackney 007E                   median=6  mean=5.6  P25=5  P75=6
  Islington 001C                 median=7  mean=6.9  P25=7  P75=7
  Lambeth 011F                   median=8  mean=8.2  P25=8  P75=9
  Lambeth 004G                   median=9  mean=8.5  P25=8  P75=9


### 6.1. Visualisations

### Fig 1: Candidate Locations on Tube Desert Map

In [24]:
fig, ax = plt.subplots(figsize=(14, 10))

# Compute tube desert grid using ALL 358 stations
lr = np.linspace(51.29, 51.69, 200)
lo = np.linspace(-0.50, 0.33, 200)
LO, LA = np.meshgrid(lo, lr)
md = np.full(LA.shape, np.inf)
for sid in G.nodes:
    d = haversine_km(LA, LO, G.nodes[sid]['lat'], G.nodes[sid]['lon'])
    md = np.minimum(md, d)

cm = LinearSegmentedColormap.from_list('d', ['#1a9641','#a6d96a','#ffffbf','#fdae61','#d73027'], N=256)
im = ax.pcolormesh(LO, LA, md, cmap=cm, shading='gouraud', vmin=0, vmax=3)
plt.colorbar(im, ax=ax, label='Distance to nearest station (km)', shrink=0.7)

# Plot all 358 stations
ax.scatter([G.nodes[n]['lon'] for n in G.nodes], [G.nodes[n]['lat'] for n in G.nodes],
           c='white', edgecolors='#333', s=8, zorder=5, linewidths=0.3)

# Plot candidates by MCDA rank
for _, r in rdf.iterrows():
    c = '#FFD700' if r['mcda_rank'] <= 3 else '#FF6B6B' if r['mcda_rank'] <= 6 else '#87CEEB'
    sz = 250 if r['mcda_rank'] <= 3 else 150
    ax.scatter(r['lon'], r['lat'], c=c, edgecolors='black', s=sz, zorder=10,
              linewidths=1.5, marker='*')
    ax.annotate(f"#{r['mcda_rank']} {r['lsoa_name']}", (r['lon'], r['lat']),
               fontsize=7, fontweight='bold', xytext=(8, 8), textcoords='offset points',
               bbox=dict(boxstyle='round,pad=0.2', facecolor=c, alpha=0.85))

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Candidate Locations on Tube Desert Map (358-station network)\n'
             'Gold = top 3, Red = 4-6, Blue = 7+')
plt.tight_layout()
plt.show()

### Fig 2: MCDA Score Decomposition


In [25]:
fig, ax = plt.subplots(figsize=(14, 8))
cols_bar = plt.cm.Set3(np.linspace(0, 1, len(ccols)))
bot = np.zeros(len(rdf))
for j, c in enumerate(ccols):
    cont = rdf[f'{c}_n'].values * weights[c]
    ax.barh(rdf['lsoa_name'].values, cont, left=bot, color=cols_bar[j],
            label=criteria[c][2], edgecolor='white', linewidth=0.3)
    bot += cont
for i, (_, r) in enumerate(rdf.iterrows()):
    ax.text(r['mcda'] + 0.003, i, f"{r['mcda']:.4f}", va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('MCDA Composite Score')
ax.set_title('MCDA Score Decomposition by Criterion')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
ax.invert_yaxis(); plt.tight_layout(); plt.show()

### Fig 3: Multi-Criteria Radar (Top 5)

In [26]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, polar=True)
ang = np.linspace(0, 2*np.pi, len(ccols), endpoint=False).tolist(); ang += ang[:1]
rc = ['#FFD700', '#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
for idx, (_, r) in enumerate(rdf.head(5).iterrows()):
    v = [r[f'{c}_n'] for c in ccols]; v += v[:1]
    ax.plot(ang, v, 'o-', lw=2, label=r['lsoa_name'], color=rc[idx], ms=4)
    ax.fill(ang, v, alpha=0.08, color=rc[idx])
ax.set_xticks(ang[:-1])
ax.set_xticklabels([criteria[c][2].replace(' ', '\n') for c in ccols], fontsize=7)
ax.set_ylim(0, 1); ax.set_title('Top 5 Candidates: Multi-Criteria Radar', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1), fontsize=8)
plt.tight_layout(); plt.show()

### Fig 4: Betweenness Centrality Change (Top 4)

In [27]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for idx, (_, cr) in enumerate(rdf.head(4).iterrows()):
    ax = axes[idx // 2][idx % 2]
    Gt = G.copy(); nn = cr['nn']
    Gt.add_node(nn, lat=cr['lat'], lon=cr['lon'], name=cr['lsoa_name'])
    for sid in cr['connected_ids']:
        if sid in Gt.nodes:
            d = haversine_km(cr['lat'], cr['lon'], Gt.nodes[sid]['lat'], Gt.nodes[sid]['lon'])
            Gt.add_edge(nn, sid, weight=d)
    bct = nx.betweenness_centrality(Gt, weight='weight')
    pos = {n: (G.nodes[n]['lon'], G.nodes[n]['lat']) for n in G.nodes}
    pos[nn] = (cr['lon'], cr['lat'])
    ch = [bct.get(n, 0) - bc_base.get(n, 0) for n in G.nodes]
    vm = max(abs(min(ch)), abs(max(ch)), 0.001)
    nx.draw_networkx_edges(Gt, pos, ax=ax, alpha=0.1, width=0.4)
    nd = nx.draw_networkx_nodes(G, pos, ax=ax, node_size=15, node_color=ch,
                                 cmap='RdYlGn_r', vmin=-vm, vmax=vm)
    ax.scatter(cr['lon'], cr['lat'], c='#FFD700', edgecolors='black',
              s=200, zorder=10, marker='*', linewidths=2)
    ax.set_title(f"#{cr['mcda_rank']} {cr['lsoa_name']}\nLines: {', '.join(cr['lines'][:3])}", fontsize=10)
    ax.set_aspect('equal'); plt.colorbar(nd, ax=ax, label='ΔBC', shrink=0.5)
plt.suptitle('Betweenness Centrality Change (Green = relieved, Red = increased)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### Fig 5: Sensitivity Analysis


In [28]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))

sw = sorted(wins.items(), key=lambda x: -x[1])
a1.barh([x[0] for x in sw], [x[1]/nsim*100 for x in sw],
        color=['#FFD700' if x[1] == max(v for _, v in sw) else '#87CEEB' for x in sw])
for i, (n, c) in enumerate(sw):
    a1.text(c/nsim*100 + 0.5, i, f'{c/nsim*100:.1f}%', va='center', fontsize=9, fontweight='bold')
a1.set_xlabel('Win Frequency (%)'); a1.invert_yaxis()
a1.set_title(f'Win Frequency (±30%, {nsim:,} sims)')

rd2 = [rdist[n] for n in rdf['lsoa_name']]
bp = a2.boxplot(rd2, vert=False, labels=rdf['lsoa_name'].values, patch_artist=True,
                medianprops=dict(color='red', lw=2))
for p, (_, r) in zip(bp['boxes'], rdf.iterrows()):
    p.set_facecolor('#FFD700' if r['mcda_rank'] <= 3 else '#87CEEB'); p.set_alpha(0.7)
a2.set_xlabel('Rank'); a2.set_title('Rank Distribution'); a2.invert_yaxis()

plt.suptitle('Sensitivity Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### Fig 6: Individual Metric Comparison


In [ ]:
TOP_N = 20  

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
mplots = [
    ('dge', 'Global Efficiency Δ (%)', '#3498DB'),
    ('bottleneck_relief', 'Bottleneck Relief', '#E74C3C'),
    ('imp_pairs', 'Improved OD Pairs', '#2ECC71'),
    ('cpop', 'Catchment Pop (800m)', '#9B59B6'),
    ('cimd', 'Avg Deprivation (IMD)', '#F39C12'),
    ('need_score', 'Your Need Score', '#1ABC9C'),
]
for ax, (col, t, clr) in zip(axes.flatten(), mplots):
    sd = rdf.sort_values(col, ascending=False).head(TOP_N) 
    ax.barh(sd['lsoa_name'], sd[col], color=clr, alpha=0.8, edgecolor='white')
    ax.set_title(t, fontsize=11)
    ax.invert_yaxis()
    for i, v in enumerate(sd[col]):
        ax.text(v, i, f' {v:.3f}', va='center', fontsize=7)

plt.suptitle('Impact Metrics Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

: 

### Export Results

In [ ]:
ecols = ['mcda_rank','lsoa_name','lsoa_code','lat','lon','need_score','rank',
         'lines','connected','n_conn','mcda','dge','bottleneck_relief','bc_new',
         'imp_pairs','dapl','path_sav','cpop','clsoas','cimd','ge_new','apl_new']
edf = rdf[[c for c in ecols if c in rdf.columns]].copy()
for c in ['lines', 'connected']:
    if c in edf.columns:
        edf[c] = edf[c].apply(lambda x: '; '.join(str(i) for i in x) if isinstance(x, list) else str(x))
edf.to_csv('station_impact_results.csv', index=False)
edf.to_excel('station_impact_results.xlsx', index=False)
print("✓ Results exported")

w = rdf.iloc[0]
print(f"\n{'='*60}")
print(f"🏆 TOP RECOMMENDATION: {w['lsoa_name']}")
print(f"{'='*60}")
print(f"  Coordinates: ({w['lat']:.4f}, {w['lon']:.4f})")
print(f"  Lines: {', '.join(w['lines'])}")
print(f"  Connected to: {', '.join(str(s) for s in w['connected'])}")
print(f"  MCDA Score: {w['mcda']:.4f}")
print(f"  Need Score: {w['need_score']:.4f}")
print(f"  ΔGlobal Efficiency: {w['dge']:+.4f}%")
print(f"  Bottleneck Relief: {w['bottleneck_relief']:+.4f}")
print(f"  Catchment Pop: {w['cpop']:,.0f}")
print(f"  Sensitivity Win Rate: {wins.get(w['lsoa_name'], 0)/nsim*100:.1f}%")

## Section 7: Interactive Map (Stage 7)

- Folium interactive map with layers: existing stations, candidate sites, LSOA choropleths
- Pop-up cards for each candidate with key metrics
- Toggle layers for different scoring dimensions
- Export as standalone HTML for presentation

## Section 8: Conclusions & Recommendation (Stage 8)

- Final recommendation with supporting evidence
- Summary table of top candidates with all metrics
- Limitations and caveats
- Suggestions for further analysis (e.g., land use data, cost estimation, engineering feasibility)